In [12]:
"""
拓扑优化PINN相变储热系统 - 重构版
基于：Heat transfer characteristics of topological latent heat storage systems based on optimization objectives
X. Zhang et al. Applied Thermal Engineering 252 (2024) 123674
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import time
from datetime import datetime
import json

# 设置随机种子保证可重复性
torch.manual_seed(42)
np.random.seed(42)

class PCMProperties:
    """相变材料属性类 - 基于论文Table 1"""
    def __init__(self):
        # Paraffin properties
        self.rho_paraffin_s = 880.0      # kg/m³
        self.rho_paraffin_l = 760.0      # kg/m³
        self.cp_paraffin_s = 2180.0      # J/(kg·K)
        self.cp_paraffin_l = 2390.0      # J/(kg·K)
        self.lambda_paraffin_s = 0.4     # W/(m·K)
        self.lambda_paraffin_l = 0.15    # W/(m·K)
        self.mu_paraffin = 0.001         # kg/(m·s)
        self.L_paraffin = 255000.0       # J/kg (255 kJ/kg)
        self.T_pc = 316.15               # K (相变温度)
        self.delta_T = 6.0               # K (相变区间)
        
        # Copper properties
        self.rho_copper = 8960.0         # kg/m³
        self.cp_copper = 385.0           # J/(kg·K)
        self.lambda_copper = 400.0       # W/(m·K)
        
        # 几何参数 (论文Fig. 3)
        self.d0 = 0.025                  # m (内径25mm - 根据论文d0=50mm直径)
        self.d1 = 0.065                  # m (外径65mm - 根据论文d1=130mm直径)
        
        # 物理参数
        self.g = 9.81                    # m/s²
        self.beta = 3.4e-4               # 热膨胀系数 1/K (石蜡典型值)
        
        # 优化参数
        self.volume_fraction_target = 0.3  # 体积分数约束
        self.A_m = 1e5                   # 糊状区常数

class PhaseChangeModel:
    """相变模型"""
    @staticmethod
    def liquid_fraction(T, T_pc, delta_T):
        """计算液相率"""
        return 0.5 * (1.0 + torch.tanh((T - T_pc) / (delta_T / 2.0)))
    
    @staticmethod
    def delta_function(T, T_pc, delta_T):
        """计算δ函数"""
        return torch.exp(-((T - T_pc) / (delta_T / 4.0))**2) / \
               (np.sqrt(np.pi) * (delta_T / 4.0))

class MaterialProperties:
    """材料属性计算"""
    @staticmethod
    def compute_properties(T, rho_a, props):
        """计算材料属性"""
        # 液相率
        phi = PhaseChangeModel.liquid_fraction(T, props.T_pc, props.delta_T)
        
        # PCM密度
        rho_pcm = props.rho_paraffin_s + (props.rho_paraffin_l - props.rho_paraffin_s) * phi
        
        # 混合密度
        rho_mix = rho_a * props.rho_copper + (1 - rho_a) * rho_pcm
        
        # PCM导热系数
        lambda_pcm = props.lambda_paraffin_s + (props.lambda_paraffin_l - props.lambda_paraffin_s) * phi
        
        # 混合导热系数
        lambda_mix = rho_a * props.lambda_copper + (1 - rho_a) * lambda_pcm
        
        # PCM比热
        D_T = PhaseChangeModel.delta_function(T, props.T_pc, props.delta_T)
        cp_pcm = props.cp_paraffin_s + phi * (props.cp_paraffin_l - props.cp_paraffin_s) + \
                props.L_paraffin * D_T
        
        # 混合比热
        cp_mix = rho_a * props.cp_copper + (1 - rho_a) * cp_pcm
        
        return {
            'rho_mix': rho_mix,
            'lambda_mix': lambda_mix,
            'cp_mix': cp_mix,
            'phi': phi,
            'D_T': D_T
        }

class PhysicsInformedNN(nn.Module):
    """物理信息神经网络 - 重构版"""
    def __init__(self, props):
        super().__init__()
        self.props = props
        
        # 温度场网络 - 简化但有效的结构
        self.temp_net = nn.Sequential(
            nn.Linear(3, 32),
            nn.Tanh(),
            nn.Linear(32, 64),
            nn.Tanh(),
            nn.Linear(64, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )
        
        # 设计变量网络 - 关键修改：添加更强的正则化
        self.design_net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Dropout(0.1),  # 添加dropout防止过拟合
            nn.Linear(32, 64),
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
        
        # 初始化权重 - 确保设计变量初始值接近0.3
        self._init_weights()
    
    def _init_weights(self):
        """初始化网络权重 - 确保设计变量初始值接近目标值"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.5)
                if m.bias is not None:
                    if m is self.design_net[-2]:  # 设计网络倒数第二层
                        nn.init.constant_(m.bias, -0.5)  # 使初始输出接近0.3
                    elif m is self.temp_net[-1]:  # 温度网络最后一层
                        nn.init.constant_(m.bias, self.props.T_pc)
                    else:
                        nn.init.zeros_(m.bias)
    
    def forward(self, x, y, t):
        """前向传播"""
        # 组合输入
        coords = torch.cat([x, y, t], dim=1)
        
        # 预测温度场
        T = self.temp_net(coords)
        
        # 预测设计变量
        design_coords = torch.cat([x, y], dim=1)
        rho_a = self.design_net(design_coords)
        
        return T, rho_a
    
    def compute_energy_residual(self, x, y, t, rho_a=None):
        """计算能量方程残差"""
        # 确保输入需要梯度
        x = x.clone().requires_grad_(True)
        y = y.clone().requires_grad_(True)
        t = t.clone().requires_grad_(True)
        
        # 前向传播
        T, rho_a_pred = self.forward(x, y, t)
        if rho_a is None:
            rho_a = rho_a_pred
        
        # 计算材料属性
        material = MaterialProperties.compute_properties(T, rho_a, self.props)
        
        # 计算一阶导数
        T_x = torch.autograd.grad(T, x, grad_outputs=torch.ones_like(T), 
                                 create_graph=True, retain_graph=True)[0]
        T_y = torch.autograd.grad(T, y, grad_outputs=torch.ones_like(T), 
                                 create_graph=True, retain_graph=True)[0]
        T_t = torch.autograd.grad(T, t, grad_outputs=torch.ones_like(T), 
                                 create_graph=True, retain_graph=True)[0]
        
        # 计算二阶导数
        T_xx = torch.autograd.grad(T_x, x, grad_outputs=torch.ones_like(T_x), 
                                  create_graph=True, retain_graph=True)[0]
        T_yy = torch.autograd.grad(T_y, y, grad_outputs=torch.ones_like(T_y), 
                                  create_graph=True, retain_graph=True)[0]
        
        laplacian_T = T_xx + T_yy
        
        # 能量方程残差 (导热为主，忽略对流以简化)
        energy_residual = (
            material['rho_mix'] * material['cp_mix'] * T_t -
            material['lambda_mix'] * laplacian_T
        )
        
        return energy_residual, T, rho_a, material, T_x, T_y, laplacian_T

class TopologyOptimizationTrainer:
    """拓扑优化训练器 - 重构版"""
    def __init__(self, model, pcm_props, case='case1'):
        self.model = model
        self.props = pcm_props
        self.case = case
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        
        # 分离优化器：先训练设计网络，再联合训练
        self.design_optimizer = torch.optim.Adam(
            model.design_net.parameters(),
            lr=0.001,
            weight_decay=0.01  # 更强的正则化
        )
        
        self.temp_optimizer = torch.optim.Adam(
            model.temp_net.parameters(),
            lr=0.0005
        )
        
        # 联合优化器
        self.joint_optimizer = torch.optim.Adam(
            list(model.temp_net.parameters()) + list(model.design_net.parameters()),
            lr=0.0002
        )
        
        # 损失历史
        self.history = {
            'total_loss': [], 'energy_loss': [], 'bc_loss': [], 
            'initial_loss': [], 'volume_loss': [], 'binary_loss': [],
            'temperature_mean': [], 'temperature_std': [], 
            'volume_frac': [], 'binary_score': [], 'phi_mean': []
        }
        
        # 创建结果目录
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.result_dir = f"./results_{case}_{timestamp}"
        os.makedirs(self.result_dir, exist_ok=True)
        
        # 保存配置
        self._save_config()
    
    def _save_config(self):
        """保存训练配置"""
        config = {
            'case': self.case,
            'volume_fraction_target': self.props.volume_fraction_target,
            'T_pc': self.props.T_pc,
            'delta_T': self.props.delta_T,
            'd0': self.props.d0,
            'd1': self.props.d1,
            'device': str(self.device)
        }
        
        with open(os.path.join(self.result_dir, 'config.json'), 'w') as f:
            json.dump(config, f, indent=2)
    
    def generate_training_data(self, n_points=2000, process='charging'):
        """生成训练数据"""
        # 内部点
        theta = torch.rand(n_points, 1) * 2 * np.pi
        r = self.props.d0 + (self.props.d1 - self.props.d0) * torch.rand(n_points, 1)
        x_int = r * torch.cos(theta)
        y_int = r * torch.sin(theta)
        t_int = torch.rand(n_points, 1) * 10000  # 减小时间范围
        
        # 边界点 (内壁)
        n_bc = 200
        theta_bc = torch.rand(n_bc, 1) * 2 * np.pi
        x_bc_inner = self.props.d0 * torch.cos(theta_bc)
        y_bc_inner = self.props.d0 * torch.sin(theta_bc)
        t_bc = torch.rand(n_bc, 1) * 10000
        
        # 边界点 (外壁)
        x_bc_outer = self.props.d1 * torch.cos(theta_bc)
        y_bc_outer = self.props.d1 * torch.sin(theta_bc)
        
        # 初始条件点
        n_ic = 200
        theta_ic = torch.rand(n_ic, 1) * 2 * np.pi
        r_ic = self.props.d0 + (self.props.d1 - self.props.d0) * torch.rand(n_ic, 1)
        x_ic = r_ic * torch.cos(theta_ic)
        y_ic = r_ic * torch.sin(theta_ic)
        t_ic = torch.zeros(n_ic, 1)
        
        # 组合所有点
        x_all = torch.cat([x_int, x_bc_inner, x_bc_outer, x_ic], dim=0)
        y_all = torch.cat([y_int, y_bc_inner, y_bc_outer, y_ic], dim=0)
        t_all = torch.cat([t_int, t_bc, t_bc, t_ic], dim=0)
        
        # 创建掩码
        n_int = len(x_int)
        
        masks = {
            'interior': torch.cat([
                torch.ones(n_int, 1),
                torch.zeros(n_bc, 1),
                torch.zeros(n_bc, 1),
                torch.zeros(n_ic, 1)
            ], dim=0).bool(),
            'bc_inner': torch.cat([
                torch.zeros(n_int, 1),
                torch.ones(n_bc, 1),
                torch.zeros(n_bc, 1),
                torch.zeros(n_ic, 1)
            ], dim=0).bool(),
            'bc_outer': torch.cat([
                torch.zeros(n_int, 1),
                torch.zeros(n_bc, 1),
                torch.ones(n_bc, 1),
                torch.zeros(n_ic, 1)
            ], dim=0).bool(),
            'initial': torch.cat([
                torch.zeros(n_int, 1),
                torch.zeros(n_bc, 1),
                torch.zeros(n_bc, 1),
                torch.ones(n_ic, 1)
            ], dim=0).bool()
        }
        
        # 边界条件目标值
        if process == 'charging':
            T_bc_target = 360.0
            T_initial_target = 290.0
        else:
            T_bc_target = 290.0
            T_initial_target = 360.0
        
        # 转移到设备
        data = {
            'x': x_all.to(self.device),
            'y': y_all.to(self.device),
            't': t_all.to(self.device),
            'masks': {k: v.to(self.device) for k, v in masks.items()},
            'T_bc_target': T_bc_target,
            'T_initial_target': T_initial_target,
            'process': process
        }
        
        return data
    
    def compute_design_losses(self, rho_a):
        """仅计算设计相关的损失"""
        # 体积约束损失 - 使用铰链损失
        volume_frac = torch.mean(rho_a)
        volume_error = volume_frac - self.props.volume_fraction_target
        volume_loss = 1000.0 * F.relu(torch.abs(volume_error) - 0.05)**2
        
        # 二元化惩罚 - 鼓励接近0或1
        binary_loss = 100.0 * torch.mean(rho_a * (1.0 - rho_a))
        
        # 多样性惩罚 - 防止所有值相同
        diversity_loss = 10.0 * (1.0 - torch.std(rho_a))
        
        total_loss = volume_loss + binary_loss + diversity_loss
        
        metrics = {
            'volume_loss': volume_loss.item(),
            'binary_loss': binary_loss.item(),
            'diversity_loss': diversity_loss.item(),
            'volume_frac': volume_frac.item(),
            'binary_score': torch.mean((rho_a > 0.5).float()).item()
        }
        
        return total_loss, metrics
    
    def compute_physics_losses(self, data, rho_a=None):
        """计算物理相关的损失"""
        # 计算能量方程残差
        energy_residual, T, rho_a_pred, material, T_x, T_y, laplacian_T = \
            self.model.compute_energy_residual(data['x'], data['y'], data['t'], rho_a)
        
        if rho_a is None:
            rho_a = rho_a_pred
        
        # 能量方程损失
        energy_loss = torch.mean(energy_residual**2)
        
        # 边界条件损失
        T_bc_inner = T[data['masks']['bc_inner'].squeeze()]
        if T_bc_inner.numel() > 0:
            bc_inner_loss = torch.mean((T_bc_inner - data['T_bc_target'])**2)
        else:
            bc_inner_loss = torch.tensor(0.0, device=self.device)
        
        # 外壁绝热损失
        bc_outer_mask = data['masks']['bc_outer'].squeeze()
        if bc_outer_mask.any():
            T_x_bc = T_x[bc_outer_mask]
            T_y_bc = T_y[bc_outer_mask]
            x_bc = data['x'][bc_outer_mask]
            y_bc = data['y'][bc_outer_mask]
            
            r = torch.sqrt(x_bc**2 + y_bc**2 + 1e-8)
            cos_theta = x_bc / r
            sin_theta = y_bc / r
            
            T_r = T_x_bc * cos_theta + T_y_bc * sin_theta
            bc_outer_loss = torch.mean(T_r**2)
        else:
            bc_outer_loss = torch.tensor(0.0, device=self.device)
        
        bc_loss = bc_inner_loss + 0.1 * bc_outer_loss
        
        # 初始条件损失
        T_initial = T[data['masks']['initial'].squeeze()]
        if T_initial.numel() > 0:
            initial_loss = torch.mean((T_initial - data['T_initial_target'])**2)
        else:
            initial_loss = torch.tensor(0.0, device=self.device)
        
        # 根据case计算目标函数
        T_mean = torch.mean(T)
        T_std = torch.std(T)
        
        # 火积耗散
        entransy_dissipation = torch.mean(material['lambda_mix'] * laplacian_T**2)
        
        # 根据case调整目标函数
        if self.case == 'case1':
            objective_loss = 0.001 * T_mean
        elif self.case == 'case2':
            objective_loss = 0.001 * T_std
        elif self.case == 'case3':
            objective_loss = 0.001 * T_mean + 0.001 * T_std + 0.0001 * entransy_dissipation
        else:
            objective_loss = 0.0
        
        total_loss = (
            1.0 * energy_loss +
            10.0 * bc_loss +
            5.0 * initial_loss +
            objective_loss
        )
        
        metrics = {
            'energy_loss': energy_loss.item(),
            'bc_loss': bc_loss.item(),
            'initial_loss': initial_loss.item(),
            'temperature_mean': T_mean.item(),
            'temperature_std': T_std.item(),
            'entransy_loss': entransy_dissipation.item(),
            'phi_mean': torch.mean(material['phi']).item()
        }
        
        return total_loss, metrics, T, rho_a, material
    
    def train(self, epochs=300, batch_size=128):
        """训练模型 - 分阶段训练策略"""
        print(f"开始训练 {self.case}...")
        print(f"设备: {self.device}")
        print(f"目标体积分数: {self.props.volume_fraction_target}")
        print("=" * 60)
        
        # 生成训练数据
        data = self.generate_training_data(process='charging')
        
        # 阶段1：仅训练设计网络，建立基本拓扑
        print("\n阶段1：训练设计网络...")
        design_data = {
            'x': data['x'],
            'y': data['y'],
            't': torch.zeros_like(data['t'])  # 固定时间
        }
        
        for epoch in range(100):
            self.design_optimizer.zero_grad()
            
            # 前向传播
            with torch.no_grad():
                T, rho_a = self.model(design_data['x'], design_data['y'], design_data['t'])
            
            # 计算设计损失
            design_loss, design_metrics = self.compute_design_losses(rho_a)
            
            # 反向传播
            design_loss.backward()
            self.design_optimizer.step()
            
            if (epoch + 1) % 20 == 0:
                print(f"设计阶段 Epoch {epoch+1:3d}/100 | "
                      f"Volume: {design_metrics['volume_frac']:.4f} | "
                      f"Binary: {design_metrics['binary_score']:.4f}")
        
        # 阶段2：训练温度网络，固定设计变量
        print("\n阶段2：训练温度网络...")
        with torch.no_grad():
            _, fixed_rho_a = self.model(data['x'], data['y'], data['t'])
        
        for epoch in range(100):
            self.temp_optimizer.zero_grad()
            
            # 计算物理损失
            physics_loss, physics_metrics, _, _, _ = \
                self.compute_physics_losses(data, fixed_rho_a)
            
            # 反向传播
            physics_loss.backward()
            self.temp_optimizer.step()
            
            if (epoch + 1) % 20 == 0:
                print(f"温度阶段 Epoch {epoch+1:3d}/100 | "
                      f"Loss: {physics_loss.item():.4e} | "
                      f"T_mean: {physics_metrics['temperature_mean']:.2f}K")
        
        # 阶段3：联合训练
        print("\n阶段3：联合训练...")
        best_loss = float('inf')
        best_model_state = None
        
        start_time = time.time()
        
        for epoch in range(epochs):
            self.joint_optimizer.zero_grad()
            
            # 计算物理损失
            physics_loss, physics_metrics, T, rho_a, material = \
                self.compute_physics_losses(data)
            
            # 计算设计损失
            design_loss, design_metrics = self.compute_design_losses(rho_a)
            
            # 总损失
            total_loss = physics_loss + 0.1 * design_loss
            
            # 反向传播
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=0.5)
            self.joint_optimizer.step()
            
            # 记录历史
            self.history['total_loss'].append(total_loss.item())
            self.history['energy_loss'].append(physics_metrics['energy_loss'])
            self.history['bc_loss'].append(physics_metrics['bc_loss'])
            self.history['initial_loss'].append(physics_metrics['initial_loss'])
            self.history['volume_loss'].append(design_metrics['volume_loss'])
            self.history['binary_loss'].append(design_metrics['binary_loss'])
            self.history['temperature_mean'].append(physics_metrics['temperature_mean'])
            self.history['temperature_std'].append(physics_metrics['temperature_std'])
            self.history['volume_frac'].append(design_metrics['volume_frac'])
            self.history['binary_score'].append(design_metrics['binary_score'])
            self.history['phi_mean'].append(physics_metrics.get('phi_mean', 0.0))
            
            # 保存最佳模型
            if total_loss.item() < best_loss:
                best_loss = total_loss.item()
                best_model_state = self.model.state_dict().copy()
            
            # 打印进度
            if (epoch + 1) % 50 == 0:
                elapsed = time.time() - start_time
                eta = elapsed / (epoch + 1) * (epochs - epoch - 1)
                
                print(f"Epoch {epoch+1:4d}/{epochs} | "
                      f"Loss: {total_loss.item():.4e} | "
                      f"Volume: {design_metrics['volume_frac']:.4f} | "
                      f"Binary: {design_metrics['binary_score']:.4f} | "
                      f"T_mean: {physics_metrics['temperature_mean']:.2f}K | "
                      f"T_std: {physics_metrics['temperature_std']:.4f} | "
                      f"Time: {elapsed:.1f}s, ETA: {eta:.1f}s")
        
        # 加载最佳模型
        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
        
        # 保存模型
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.joint_optimizer.state_dict(),
            'loss': best_loss,
            'history': self.history
        }, os.path.join(self.result_dir, 'final_model.pth'))
        
        # 保存训练历史
        self.save_history()
        
        print(f"\n训练完成! 最佳损失: {best_loss:.4e}")
        print(f"结果保存到: {self.result_dir}")
        
        return self.history
    
    def save_history(self):
        """保存训练历史"""
        # 保存为JSON
        with open(os.path.join(self.result_dir, 'training_history.json'), 'w') as f:
            serializable_history = {}
            for key, values in self.history.items():
                if isinstance(values, list):
                    serializable_history[key] = [float(v) for v in values]
                else:
                    serializable_history[key] = float(values)
            json.dump(serializable_history, f, indent=2)
        
        # 绘制损失曲线
        self.plot_training_history()
    
    def plot_training_history(self):
        """绘制训练历史"""
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        # 总损失
        axes[0, 0].plot(self.history['total_loss'])
        axes[0, 0].set_yscale('log')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Total Loss')
        axes[0, 0].set_title('Total Loss History')
        axes[0, 0].grid(True, alpha=0.3)
        
        # PDE损失
        axes[0, 1].plot(self.history['energy_loss'], label='Energy')
        axes[0, 1].plot(self.history['bc_loss'], label='BC')
        axes[0, 1].set_yscale('log')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('PDE Loss')
        axes[0, 1].set_title('PDE Loss History')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 约束损失
        axes[0, 2].plot(self.history['volume_loss'], label='Volume')
        axes[0, 2].plot(self.history['binary_loss'], label='Binary')
        axes[0, 2].set_yscale('log')
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].set_ylabel('Constraint Loss')
        axes[0, 2].set_title('Constraint Loss History')
        axes[0, 2].legend()
        axes[0, 2].grid(True, alpha=0.3)
        
        # 体积分数和二元化分数
        axes[1, 0].plot(self.history['volume_frac'], label='Volume Fraction', color='blue')
        axes[1, 0].axhline(y=self.props.volume_fraction_target, color='r', linestyle='--', 
                          label=f'Target ({self.props.volume_fraction_target})')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Volume Fraction')
        axes[1, 0].set_title('Volume Fraction Evolution')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)
        
        axes[1, 1].plot(self.history['binary_score'], label='Binary Score', color='green')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Binary Score')
        axes[1, 1].set_title('Binary Score Evolution')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)
        
        # 温度指标
        axes[1, 2].plot(self.history['temperature_mean'], color='red', label='Mean Temp')
        axes[1, 2].plot(self.history['temperature_std'], color='orange', label='Std Temp')
        axes[1, 2].axhline(y=self.props.T_pc, color='orange', linestyle='--', 
                          label=f'Phase Change ({self.props.T_pc}K)')
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].set_ylabel('Temperature (K) / Std')
        axes[1, 2].set_title('Temperature Evolution')
        axes[1, 2].legend()
        axes[1, 2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.result_dir, 'training_history.png'), dpi=300, bbox_inches='tight')
        plt.close()
    
    def visualize_results(self):
        """可视化结果"""
        print("\n生成可视化结果...")
        
        # 创建网格
        r = torch.linspace(self.props.d0, self.props.d1, 50)
        theta = torch.linspace(0, 2*np.pi, 100)
        R, Theta = torch.meshgrid(r, theta, indexing='ij')
        
        x_grid = R * torch.cos(Theta)
        y_grid = R * torch.sin(Theta)
        
        # 展平网格
        x_flat = x_grid.reshape(-1, 1)
        y_flat = y_grid.reshape(-1, 1)
        t_flat = torch.zeros_like(x_flat)
        
        # 转移到设备
        x_flat = x_flat.to(self.device)
        y_flat = y_flat.to(self.device)
        t_flat = t_flat.to(self.device)
        
        # 计算预测
        with torch.no_grad():
            T_pred, rho_a_pred = self.model(x_flat, y_flat, t_flat)
            material = MaterialProperties.compute_properties(T_pred, rho_a_pred, self.props)
        
        # 重塑为网格形状
        T_grid = T_pred.reshape(R.shape).cpu().numpy()
        rho_a_grid = rho_a_pred.reshape(R.shape).cpu().numpy()
        phi_grid = material['phi'].reshape(R.shape).cpu().numpy()
        
        # 转换为笛卡尔坐标
        X = x_grid.cpu().numpy()
        Y = y_grid.cpu().numpy()
        
        # 创建可视化目录
        viz_dir = os.path.join(self.result_dir, 'visualization')
        os.makedirs(viz_dir, exist_ok=True)
        
        # 1. 温度分布
        fig, ax = plt.subplots(figsize=(10, 8))
        contour = ax.contourf(X, Y, T_grid, levels=50, cmap='RdYlBu_r')
        ax.set_aspect('equal')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_title(f'Temperature Distribution (K) - {self.case}')
        
        # 添加内壁和外壁
        circle_inner = plt.Circle((0, 0), self.props.d0, fill=False, 
                                 color='black', linestyle='--', linewidth=2)
        circle_outer = plt.Circle((0, 0), self.props.d1, fill=False, 
                                 color='black', linewidth=2)
        ax.add_patch(circle_inner)
        ax.add_patch(circle_outer)
        
        # 颜色条
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        plt.colorbar(contour, cax=cax)
        
        plt.savefig(os.path.join(viz_dir, 'temperature_distribution.png'), 
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        # 2. 拓扑结构
        fig, ax = plt.subplots(figsize=(10, 8))
        
        # 二元化拓扑
        rho_a_binary = (rho_a_grid > 0.5).astype(float)
        
        # 创建自定义颜色映射
        from matplotlib.colors import ListedColormap
        colors = ['#3498db', '#e74c3c']  # 蓝色表示PCM，红色表示铜
        cmap = ListedColormap(colors)
        
        contour = ax.contourf(X, Y, rho_a_binary, levels=[-0.5, 0.5, 1.5], 
                             cmap=cmap, alpha=0.8)
        
        # 添加轮廓线
        ax.contour(X, Y, rho_a_grid, levels=[0.5], colors='black', linewidths=2)
        
        ax.set_aspect('equal')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_title(f'Topology Optimization Result - {self.case}\n(Red: Copper, Blue: PCM)')
        
        # 添加内壁和外壁
        circle_inner = plt.Circle((0, 0), self.props.d0, fill=False, 
                                 color='black', linestyle='--', linewidth=2)
        circle_outer = plt.Circle((0, 0), self.props.d1, fill=False, 
                                 color='black', linewidth=2)
        ax.add_patch(circle_inner)
        ax.add_patch(circle_outer)
        
        # 添加图例
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor=colors[0], alpha=0.8, label='PCM'),
            Patch(facecolor=colors[1], alpha=0.8, label='Copper')
        ]
        ax.legend(handles=legend_elements, loc='upper right')
        
        plt.savefig(os.path.join(viz_dir, 'topology_structure.png'), 
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        # 3. 液相率分布
        fig, ax = plt.subplots(figsize=(10, 8))
        contour = ax.contourf(X, Y, phi_grid, levels=50, cmap='viridis')
        ax.set_aspect('equal')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_title(f'Liquid Fraction Distribution - {self.case}')
        
        # 添加相变温度线
        T_contour = ax.contour(X, Y, T_grid, levels=[self.props.T_pc], 
                              colors='red', linewidths=3, linestyles='--')
        ax.clabel(T_contour, inline=True, fontsize=10, fmt=f'T={self.props.T_pc}K')
        
        # 添加内壁和外壁
        circle_inner = plt.Circle((0, 0), self.props.d0, fill=False, 
                                 color='black', linestyle='--', linewidth=2)
        circle_outer = plt.Circle((0, 0), self.props.d1, fill=False, 
                                 color='black', linewidth=2)
        ax.add_patch(circle_inner)
        ax.add_patch(circle_outer)
        
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        plt.colorbar(contour, cax=cax)
        
        plt.savefig(os.path.join(viz_dir, 'liquid_fraction.png'), 
                   dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"可视化结果已保存到: {viz_dir}")
    
    def evaluate_performance(self):
        """评估性能指标"""
        print("\n分析优化结果...")
        
        # 生成评估点
        data = self.generate_training_data(process='charging')
        
        with torch.no_grad():
            T, rho_a = self.model(data['x'], data['y'], data['t'])
            material = MaterialProperties.compute_properties(T, rho_a, self.props)
        
        # 计算指标
        T_mean = torch.mean(T).item()
        T_std = torch.std(T).item()
        volume_frac = torch.mean(rho_a).item()
        binary_score = torch.mean((rho_a > 0.5).float()).item()
        phi_mean = torch.mean(material['phi']).item()
        
        # 相变利用率
        phi = material['phi']
        phase_change_mask = (phi > 0.1) & (phi < 0.9)
        phase_change_util = torch.mean(phase_change_mask.float()).item()
        
        print("=" * 60)
        print(f"性能指标 - {self.case}:")
        print(f"  平均温度: {T_mean:.2f} K")
        print(f"  温度标准差: {T_std:.4f} K")
        print(f"  实际体积分数: {volume_frac:.4f}")
        print(f"  目标体积分数: {self.props.volume_fraction_target}")
        print(f"  体积误差: {abs(volume_frac - self.props.volume_fraction_target):.4f}")
        print(f"  二元化得分: {binary_score:.3f} (1.0为完全二元化)")
        print(f"  平均液相率: {phi_mean:.3f}")
        print(f"  相变区间占比: {phase_change_util:.3f}")
        print("=" * 60)
        
        # 保存指标
        metrics = {
            'case': self.case,
            'average_temperature': T_mean,
            'temperature_std': T_std,
            'volume_fraction': volume_frac,
            'binary_score': binary_score,
            'liquid_fraction_mean': phi_mean,
            'phase_change_utilization': phase_change_util,
            'volume_target': self.props.volume_fraction_target,
            'volume_error': abs(volume_frac - self.props.volume_fraction_target)
        }
        
        with open(os.path.join(self.result_dir, 'performance_metrics.json'), 'w') as f:
            json.dump(metrics, f, indent=2)
        
        return metrics

def main():
    """主函数"""
    print("=" * 60)
    print("拓扑优化PINN相变储热系统 - 重构版")
    print("基于: Heat transfer characteristics of topological latent heat storage systems")
    print("X. Zhang et al. Applied Thermal Engineering 252 (2024) 123674")
    print("=" * 60)
    
    # 选择case
    cases = ['case1', 'case2', 'case3']
    print("可用优化目标:")
    print("  case1: 最小化平均温度")
    print("  case2: 最小化温度标准差")
    print("  case3: 多目标优化 (平均温度 + 标准差 + 火积耗散)")
    print()
    
    # 获取用户输入
    case = input("请选择case (默认: case1): ").strip() or 'case1'
    if case not in cases:
        print(f"警告: {case} 不存在，使用 case1")
        case = 'case1'
    
    # 训练参数
    try:
        epochs = int(input("输入训练轮次 (默认: 300): ").strip() or "300")
    except ValueError:
        print("输入无效，使用默认值 300")
        epochs = 300
    
    try:
        batch_size = int(input("输入批次大小 (默认: 128): ").strip() or "128")
    except ValueError:
        print("输入无效，使用默认值 128")
        batch_size = 128
    
    # 创建PCM属性
    props = PCMProperties()
    
    # 创建模型
    model = PhysicsInformedNN(props)
    
    # 验证初始模型
    print("\n验证初始模型...")
    print(f"相变温度: {props.T_pc} K")
    print(f"相变区间: {props.delta_T} K")
    print(f"内径: {props.d0} m, 外径: {props.d1} m")
    print(f"目标体积分数: {props.volume_fraction_target}")
    print(f"使用设备: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
    print("=" * 60)
    
    # 创建训练器
    trainer = TopologyOptimizationTrainer(model, props, case=case)
    
    # 训练模型
    history = trainer.train(epochs=epochs, batch_size=batch_size)
    
    # 可视化和评估
    trainer.visualize_results()
    metrics = trainer.evaluate_performance()
    
    print(f"\n程序执行完成!")
    print(f"结果保存目录: {trainer.result_dir}")
    print("=" * 60)

if __name__ == "__main__":
    main()

拓扑优化PINN相变储热系统 - 重构版
基于: Heat transfer characteristics of topological latent heat storage systems
X. Zhang et al. Applied Thermal Engineering 252 (2024) 123674
可用优化目标:
  case1: 最小化平均温度
  case2: 最小化温度标准差
  case3: 多目标优化 (平均温度 + 标准差 + 火积耗散)



请选择case (默认: case1):  
输入训练轮次 (默认: 300):  
输入批次大小 (默认: 128):  



验证初始模型...
相变温度: 316.15 K
相变区间: 6.0 K
内径: 0.025 m, 外径: 0.065 m
目标体积分数: 0.3
使用设备: cuda
开始训练 case1...
设备: cuda
目标体积分数: 0.3

阶段1：训练设计网络...


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn